In [ ]:
%pip install pandas openpyxl ipykernel

In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

PASTA_DADOS = Path.cwd() / "data" / "brutos" / "ssp"

if not PASTA_DADOS.exists():
    PASTA_DADOS = Path.cwd().parent / "data" / "brutos" / "ssp"

print("Pasta dos dados:", PASTA_DADOS)
print("Arquivos encontrados:")
for arquivo in PASTA_DADOS.glob("*.xlsx"):
    print("-", arquivo.name)
    

In [ ]:
arquivos = [
    "SPDadosCriminais_2026.xlsx",
    "CelularesSubtraidos_2026.xlsx",
    "ObjetosSubtraidos_2026.xlsx",
]

for nome in arquivos:
    caminho = PASTA_DADOS / nome

    if not caminho.exists():
        print(f"\nArquivo não encontrado: {nome}")
        continue

    excel = pd.ExcelFile(caminho)

    print(f"\nArquivo: {nome}")
    print("Abas:", excel.sheet_names)
    excel.close()

In [ ]:
nome = "ObjetosSubtraidos_2026.xlsx"
caminho = PASTA_DADOS / nome

with pd.ExcelFile(caminho) as excel:
    print("Abas disponíveis:", excel.sheet_names)

    for aba in excel.sheet_names:
        print(f"\nAba: {aba}")
        amostra = pd.read_excel(excel, sheet_name=aba, nrows=5)
        display(amostra)
        

In [ ]:
resumo_colunas = []

for nome in arquivos:
    caminho = PASTA_DADOS / nome

    if not caminho.exists():
        continue

    with pd.ExcelFile(caminho) as excel:
        for aba in excel.sheet_names:
            amostra = pd.read_excel(
                excel,
                sheet_name=aba,
                nrows=5
            )

            for coluna in amostra.columns:
                resumo_colunas.append({
                    "arquivo": nome,
                    "aba": aba,
                    "coluna": str(coluna)
                })

df_colunas = pd.DataFrame(resumo_colunas)

display(df_colunas)

In [ ]:
palavras_chave = [
    "LAT",
    "LONG",
    "DATA",
    "HORA",
    "NATUREZA",
    "RUBRICA",
    "BAIRRO",
    "MUNICIPIO",
    "BO",
    "LOGRADOURO"
]

filtro = df_colunas["coluna"].str.upper().apply(
    lambda nome: any(palavra in nome for palavra in palavras_chave)
)

colunas_importantes = df_colunas.loc[filtro]

display(colunas_importantes)

In [ ]:
comparacao = (
    colunas_importantes
    .groupby(["arquivo", "aba"])["coluna"]
    .apply(lambda colunas: ", ".join(colunas))
    .reset_index()
)

display(comparacao)


In [ ]:
from openpyxl import load_workbook
import pandas as pd

abas_dados = {
    "SPDadosCriminais_2026.xlsx": [
        "JAN-JUN_2026",
        "JUL-DEZ_2026"
    ],
    "CelularesSubtraidos_2026.xlsx": [
        "CELULAR_2026"
    ],
    "ObjetosSubtraidos_2026.xlsx": [
        "OBJETOS_01_TRIM_2026",
        "OBJETOS_02_TRIM_2026",
        "OBJETOS_03_TRIM_2026"
    ]
}

resumo = []

for nome_arquivo, abas in abas_dados.items():
    caminho = PASTA_DADOS / nome_arquivo

    if not caminho.exists():
        print(f"Arquivo não encontrado: {nome_arquivo}")
        continue

    wb = load_workbook(caminho, read_only=True, data_only=True)

    try:
        for nome_aba in abas:
            if nome_aba not in wb.sheetnames:
                print(f"Aba não encontrada: {nome_aba}")
                continue

            ws = wb[nome_aba]

            resumo.append({
                "arquivo": nome_arquivo,
                "aba": nome_aba,
                "linhas_estimadas": ws.max_row - 1,
                "colunas": ws.max_column
            })
    finally:
        wb.close()

df_resumo = pd.DataFrame(resumo)

display(df_resumo)

In [ ]:
campos_essenciais = [
    "NUM_BO",
    "ANO_BO",
    "DATA_OCORRENCIA_BO",
    "RUBRICA",
    "BAIRRO",
    "LATITUDE",
    "LONGITUDE"
]

verificacao = []

for nome_arquivo, abas in abas_dados.items():
    caminho = PASTA_DADOS / nome_arquivo

    if not caminho.exists():
        continue

    with pd.ExcelFile(caminho) as excel:
        for nome_aba in abas:
            colunas = pd.read_excel(
                excel,
                sheet_name=nome_aba,
                nrows=0
            ).columns

            verificacao.append({
                "arquivo": nome_arquivo,
                "aba": nome_aba,
                **{
                    campo: campo in colunas
                    for campo in campos_essenciais
                }
            })

df_verificacao = pd.DataFrame(verificacao)

display(df_verificacao)

In [ ]:
for nome_arquivo in [
    "CelularesSubtraidos_2026.xlsx",
    "ObjetosSubtraidos_2026.xlsx"
]:
    caminho = PASTA_DADOS / nome_arquivo

    print(f"\n{'=' * 60}")
    print(nome_arquivo)

    metodologia = pd.read_excel(
        caminho,
        sheet_name="METODOLOGIA",
        header=None
    )

    for linha in metodologia.itertuples(index=False, name=None):
        for valor in linha:
            if pd.notna(valor):
                print(str(valor))

In [ ]:
nome_arquivo = "CelularesSubtraidos_2026.xlsx"
nome_aba = "CELULAR_2026"

colunas = [
    "NOME_DELEGACIA",
    "ANO_BO",
    "NUM_BO"
]

amostra = pd.read_excel(
    PASTA_DADOS / nome_arquivo,
    sheet_name=nome_aba,
    usecols=colunas,
    nrows=10000
)

total_linhas = len(amostra)

boletins_unicos = (
    amostra
    .dropna(subset=colunas)
    .drop_duplicates(subset=colunas)
)

print("Linhas analisadas:", total_linhas)
print("Boletins distintos na amostra:", len(boletins_unicos))
print("Linhas com identificação incompleta:", amostra[colunas].isna().any(axis=1).sum())
print("Linhas adicionais com identificação repetida:", len(amostra.dropna(subset=colunas)) - len(boletins_unicos))

In [ ]:
frequencia = (
    amostra
    .dropna(subset=colunas)
    .groupby(colunas)
    .size()
    .value_counts()
    .sort_index()
    .reset_index()
)

frequencia.columns = [
    "linhas_por_boletim",
    "quantidade_de_boletins"
]

display(frequencia)

In [ ]:
colunas_tri2 = pd.read_excel(
    PASTA_DADOS / "ObjetosSubtraidos_2026.xlsx",
    sheet_name="OBJETOS_02_TRIM_2026",
    nrows=0
).columns.tolist()

colunas_tri3 = pd.read_excel(
    PASTA_DADOS / "ObjetosSubtraidos_2026.xlsx",
    sheet_name="OBJETOS_03_TRIM_2026",
    nrows=0
).columns.tolist()

print("Exclusivas do terceiro trimestre:")
print(set(colunas_tri3) - set(colunas_tri2))

print("\nExclusivas do segundo trimestre:")
print(set(colunas_tri2) - set(colunas_tri3))

In [ ]:
resumo_amostras = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["DATA_OCORRENCIA_BO", "LATITUDE", "LONGITUDE"],
            nrows=10000
        )

        datas = pd.to_datetime(
            amostra["DATA_OCORRENCIA_BO"],
            errors="coerce",
            dayfirst=True
        )

        lat = pd.to_numeric(amostra["LATITUDE"], errors="coerce")
        lon = pd.to_numeric(amostra["LONGITUDE"], errors="coerce")

        coordenadas_validas = (
            lat.between(-90, 90)
            & lon.between(-180, 180)
            & (lat != 0)
            & (lon != 0)
        )

        resumo_amostras.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "linhas_amostra": len(amostra),
            "datas_ausentes_ou_invalidas": int(datas.isna().sum()),
            "data_minima": datas.min(),
            "data_maxima": datas.max(),
            "coordenadas_basicas_validas": int(coordenadas_validas.sum()),
            "coordenadas_ausentes_ou_invalidas": int((~coordenadas_validas).sum())
        })

df_resumo_amostras = pd.DataFrame(resumo_amostras)

display(df_resumo_amostras)

In [ ]:
amostra_crimes = pd.read_excel(
    PASTA_DADOS / "SPDadosCriminais_2026.xlsx",
    sheet_name="JAN-JUN_2026",
    usecols=["RUBRICA", "NATUREZA_APURADA"],
    nrows=10000
)

print("Rubricas mais frequentes na amostra:")
display(amostra_crimes["RUBRICA"].value_counts(dropna=False).head(20))

print("\nNaturezas apuradas mais frequentes:")
display(amostra_crimes["NATUREZA_APURADA"].value_counts(dropna=False).head(20))

In [ ]:
resumo_anos = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["DATA_OCORRENCIA_BO"],
            nrows=10000
        )

        datas = pd.to_datetime(
            amostra["DATA_OCORRENCIA_BO"],
            errors="coerce",
            dayfirst=True
        )

        resumo_anos.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "antes_de_2026": int((datas.dt.year < 2026).sum()),
            "em_2026": int((datas.dt.year == 2026).sum()),
            "apos_2026": int((datas.dt.year > 2026).sum()),
            "datas_invalidas": int(datas.isna().sum())
        })

display(pd.DataFrame(resumo_anos))

In [ ]:
resumo_localizacao = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["LATITUDE", "LONGITUDE"],
            nrows=10000
        )

        lat = pd.to_numeric(amostra["LATITUDE"], errors="coerce")
        lon = pd.to_numeric(amostra["LONGITUDE"], errors="coerce")

        plausivel_sp = (
            lat.between(-24.1, -23.3)
            & lon.between(-47.0, -46.3)
        )

        resumo_localizacao.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "amostra": len(amostra),
            "coordenadas_na_area_aproximada": int(plausivel_sp.sum()),
            "fora_da_area_ou_invalidas": int((~plausivel_sp).sum())
        })

display(pd.DataFrame(resumo_localizacao))

In [ ]:
resumo_localidades = []

for nome_arquivo, abas in abas_dados.items():
    for aba in abas:
        amostra = pd.read_excel(
            PASTA_DADOS / nome_arquivo,
            sheet_name=aba,
            usecols=["NOME_MUNICIPIO", "BAIRRO"],
            nrows=10000
        )

        municipio = (
            amostra["NOME_MUNICIPIO"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        bairro = (
            amostra["BAIRRO"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        resumo_localidades.append({
            "arquivo": nome_arquivo,
            "aba": aba,
            "linhas_amostra": len(amostra),
            "municipio_sao_paulo": int((municipio == "SÃO PAULO").sum()),
            "bairro_vila_mariana": int((bairro == "VILA MARIANA").sum()),
            "municipio_ausente": int(municipio.isna().sum()),
            "bairro_ausente": int(bairro.isna().sum())
        })

display(pd.DataFrame(resumo_localidades))

In [ ]:
amostra_municipios = pd.read_excel(
    PASTA_DADOS / "SPDadosCriminais_2026.xlsx",
    sheet_name="JAN-JUN_2026",
    usecols=["NOME_MUNICIPIO"],
    nrows=10000
)

display(
    amostra_municipios["NOME_MUNICIPIO"]
    .value_counts(dropna=False)
    .head(15)
)

In [ ]:
import sys
import importlib.util

print("Python do notebook:", sys.executable)
print("GeoPandas disponível:", importlib.util.find_spec("geopandas") is not None)

In [ ]:
%pip install geopandas

In [ ]:
import geopandas as gpd

print("Versão do GeoPandas:", gpd.__version__) 

In [ ]:
from pathlib import Path
import geopandas as gpd

raiz = Path.cwd()

if not (raiz / "data" / "brutos").exists():
    raiz = raiz.parent

caminho_iluminacao = (
    raiz / "data" / "brutos" / "geosampa"
    / "SIRGAS_GPKG_iluminacaopublica.gpkg"
)

if not caminho_iluminacao.is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_iluminacao}")

iluminacao = gpd.read_file(
    caminho_iluminacao,
    layer="SIRGAS_GPKG_iluminacaopublica"
)

print("Linhas e colunas:", iluminacao.shape)
print("Colunas:", iluminacao.columns.tolist())
print("Sistema de coordenadas:", iluminacao.crs)

print("\nTipos de geometria:")
print(iluminacao.geom_type.value_counts(dropna=False))

print("\nTipos das colunas:")
print(iluminacao.dtypes)

print("\nNulos por coluna:")
print(iluminacao.isna().sum())

display(iluminacao.head())

In [ ]:
colunas_texto = [
    "il_placa",
    "il_tipilum",
    "il_enderec",
    "il_unidade"
]

print("Textos vazios ou contendo apenas espaços:")
for coluna in colunas_texto:
    vazios = iluminacao[coluna].astype("string").str.strip().eq("")
    print(f"{coluna}: {int(vazios.sum())}")

print("\nGeometrias vazias:", int(iluminacao.geometry.is_empty.sum()))
print("Geometrias inválidas:", int((~iluminacao.geometry.is_valid).sum()))

print("\nTipos de iluminação:")
print(iluminacao["il_tipilum"].value_counts(dropna=False))

In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd()

if not (raiz / "src").is_dir():
    raiz = raiz.parent

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.dados import ler_ssp, ler_iluminacao

amostra_ssp = ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JAN-JUN_2026",
    colunas=["DATA_OCORRENCIA_BO", "LATITUDE", "LONGITUDE"],
    n_linhas=5
)

amostra_luz = ler_iluminacao(
    raiz / "data/brutos/geosampa/SIRGAS_GPKG_iluminacaopublica.gpkg",
    n_linhas=5
)

print("SSP — linhas e colunas:", amostra_ssp.shape)
print("Iluminação — linhas e colunas:", amostra_luz.shape)
print("Iluminação — CRS:", amostra_luz.crs)
print("Iluminação — geometrias:", amostra_luz.geom_type.tolist())

In [ ]:
import pandas as pd

fontes_dicionario = {
    "SPDadosCriminais_2026.xlsx": "Campos da Tabela_SPDADOS",
    "CelularesSubtraidos_2026.xlsx": "DICIONARIO DE DADOS",
    "ObjetosSubtraidos_2026.xlsx": "DICIONARIO DE DADOS"
}

campos_interesse = {
    "NOME_DELEGACIA", "ANO_BO", "NUM_BO", "VERSAO",
    "DATA_OCORRENCIA_BO", "DATA_REGISTRO",
    "DATAHORA_REGISTRO_BO", "RUBRICA", "NATUREZA_APURADA",
    "LATITUDE", "LONGITUDE", "LOGRADOURO", "BAIRRO",
    "NOME_MUNICIPIO", "NOME_MUNICIPIO_CIRCUNSCRICAO",
    "NOME_MUNICIPIO_CIRC", "CIDADE",
    "DESCR_TIPOLOCAL", "DESCR_SUBTIPOLOCAL"
}

for nome_arquivo, aba in fontes_dicionario.items():
    tabela = pd.read_excel(
        raiz / "data" / "brutos" / "ssp" / nome_arquivo,
        sheet_name=aba,
        header=None,
        usecols=[0, 1]
    )

    tabela.columns = ["campo", "descricao"]
    tabela["campo"] = (
        tabela["campo"].astype("string").str.strip().str.upper()
    )

    print(f"\nArquivo: {nome_arquivo}")
    print(
        tabela.loc[tabela["campo"].isin(campos_interesse)]
        .to_string(index=False)
    )
    

In [ ]:
campos_geral = [
    "NOME_DELEGACIA",
    "ANO_BO",
    "NUM_BO",
    "DATA_REGISTRO",
    "DATA_OCORRENCIA_BO",
    "RUBRICA",
    "NATUREZA_APURADA",
    "DESCR_TIPOLOCAL",
    "DESCR_SUBTIPOLOCAL",
    "BAIRRO",
    "LOGRADOURO",
    "LATITUDE",
    "LONGITUDE",
    "NOME_MUNICIPIO",
    "NOME_MUNICIPIO_CIRCUNSCRICAO"
]

amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JAN-JUN_2026",
    colunas=campos_geral,
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Arquivo: SPDadosCriminais_2026.xlsx")
print("Aba: JAN-JUN_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())

In [ ]:
amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JUL-DEZ_2026",
    colunas=campos_geral,
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Arquivo: SPDadosCriminais_2026.xlsx")
print("Aba: JUL-DEZ_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())

In [ ]:
campos_celulares_objetos = [
    "NOME_DELEGACIA",
    "ANO_BO",
    "NUM_BO",
    "VERSAO",
    "DATAHORA_REGISTRO_BO",
    "DATA_OCORRENCIA_BO",
    "RUBRICA",
    "DESCR_TIPOLOCAL",
    "DESCR_SUBTIPOLOCAL",
    "BAIRRO",
    "LOGRADOURO",
    "LATITUDE",
    "LONGITUDE",
    "NOME_MUNICIPIO",
    "NOME_MUNICIPIO_CIRC"
]

amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/CelularesSubtraidos_2026.xlsx",
    aba="CELULAR_2026",
    colunas=campos_celulares_objetos + ["CIDADE"],
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Arquivo: CelularesSubtraidos_2026.xlsx")
print("Aba: CELULAR_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())

In [ ]:
abas_objetos = [
    "OBJETOS_01_TRIM_2026",
    "OBJETOS_02_TRIM_2026",
    "OBJETOS_03_TRIM_2026"
]

for aba in abas_objetos:
    amostra_a2 = ler_ssp(
        raiz / "data/brutos/ssp/ObjetosSubtraidos_2026.xlsx",
        aba=aba,
        colunas=campos_celulares_objetos,
        n_linhas=10000
    )

    resumo_a2 = pd.DataFrame({
        "tipo_observado": amostra_a2.dtypes.astype(str),
        "nulos": amostra_a2.isna().sum(),
        "textos_em_branco": [
            int(
                amostra_a2[coluna]
                .astype("string")
                .str.strip()
                .eq("")
                .sum()
            )
            for coluna in amostra_a2.columns
        ]
    })

    print(f"\nAba: {aba}")
    print("Linhas da amostra:", len(amostra_a2))
    print(resumo_a2.to_string())

    del amostra_a2

In [ ]:
print("Iniciando leitura do terceiro trimestre...", flush=True)

amostra_a2 = ler_ssp(
    raiz / "data/brutos/ssp/ObjetosSubtraidos_2026.xlsx",
    aba="OBJETOS_03_TRIM_2026",
    colunas=campos_celulares_objetos,
    n_linhas=10000
)

resumo_a2 = pd.DataFrame({
    "tipo_observado": amostra_a2.dtypes.astype(str),
    "nulos": amostra_a2.isna().sum(),
    "textos_em_branco": [
        int(
            amostra_a2[coluna]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )
        for coluna in amostra_a2.columns
    ]
})

print("Aba: OBJETOS_03_TRIM_2026")
print("Linhas da amostra:", len(amostra_a2))
print(resumo_a2.to_string())
print("Inspeção da amostra finalizada.")

In [ ]:
import sys
from importlib import reload
from pathlib import Path

raiz = Path.cwd()
if not (raiz / "src" / "dados.py").is_file():
    raiz = raiz.parent

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

import src.dados as dados

reload(dados)

amostra_a3 = dados.ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JAN-JUN_2026",
    colunas=[
        "NOME_DELEGACIA", "ANO_BO", "NUM_BO",
        "DATA_OCORRENCIA_BO", "LATITUDE", "LONGITUDE", "RUBRICA"
    ],
    n_linhas=1000
)

amostra_padronizada = dados.padronizar_colunas(amostra_a3)

print("Colunas originais:", amostra_a3.columns.tolist())
print("Colunas padronizadas:", amostra_padronizada.columns.tolist())
print("Dimensões antes e depois:", amostra_a3.shape, amostra_padronizada.shape)

In [ ]:
import pandas as pd

amostra_convertida = amostra_padronizada.copy()

amostra_convertida["data_ocorrencia_bo"] = pd.to_datetime(
    amostra_padronizada["data_ocorrencia_bo"],
    dayfirst=True,
    errors="coerce"
)

for coluna in ["latitude", "longitude"]:
    texto = amostra_padronizada[coluna].astype("string").str.strip()
    texto = texto.str.replace(",", ".", regex=False)
    amostra_convertida[coluna] = pd.to_numeric(texto, errors="coerce")

for coluna in ["data_ocorrencia_bo", "latitude", "longitude"]:
    original = amostra_padronizada[coluna]
    convertido = amostra_convertida[coluna]

    ausentes_antes = (
        original.isna()
        | original.astype("string").str.strip().eq("").fillna(False)
    )
    falhas = (~ausentes_antes) & convertido.isna()

    print(f"\n{coluna} — {original.dtype} → {convertido.dtype}")
    print("Ausentes ou em branco antes:", int(ausentes_antes.sum()))
    print("Preenchidos que não puderam ser convertidos:", int(falhas.sum()))

print("\nDimensões após a conversão:", amostra_convertida.shape)

In [ ]:
falha_lat = amostra_convertida["latitude"].isna()
falha_lon = amostra_convertida["longitude"].isna()

print("Falha nas duas coordenadas:", int((falha_lat & falha_lon).sum()))
print("Falha só na latitude:", int((falha_lat & ~falha_lon).sum()))
print("Falha só na longitude:", int((~falha_lat & falha_lon).sum()))
print("Linhas com pelo menos uma falha:", int((falha_lat | falha_lon).sum()))

marcadores = ["null", "none", "nan", "na", "n/a", "-", "sem informação"]

for coluna, falhas in [("latitude", falha_lat), ("longitude", falha_lon)]:
    valores = amostra_padronizada.loc[falhas, coluna]
    valores = valores.astype("string").str.strip().str.lower()
    conhecidos = valores[valores.isin(marcadores)]

    print(f"\n{coluna}")
    print("Marcadores encontrados:", conhecidos.value_counts().to_dict())
    print("Outros valores a investigar:", len(valores) - len(conhecidos))

In [ ]:
latitude = amostra_convertida["latitude"]
longitude = amostra_convertida["longitude"]

tem_coordenadas = latitude.notna() & longitude.notna()
coordenada_zero = tem_coordenadas & (
    latitude.eq(0) | longitude.eq(0)
).fillna(False)

dentro_retangulo = (
    latitude.between(-24.1, -23.3)
    & longitude.between(-47.0, -46.3)
).fillna(False)

fora_retangulo = tem_coordenadas & ~coordenada_zero & ~dentro_retangulo
passaram = tem_coordenadas & ~coordenada_zero & dentro_retangulo

print("Linhas da amostra:", len(amostra_convertida))
print("Sem pelo menos uma coordenada:", int((~tem_coordenadas).sum()))
print("Com pelo menos uma coordenada igual a zero:", int(coordenada_zero.sum()))
print("Fora do retângulo aproximado:", int(fora_retangulo.sum()))
print("Dentro do retângulo aproximado:", int(passaram.sum()))

In [ ]:
amostra_completa = dados.ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JAN-JUN_2026",
    n_linhas=1000
)

# Confere se a leitura completa corresponde à amostra anterior.
if not amostra_completa[amostra_a3.columns].equals(amostra_a3):
    raise ValueError("A amostra completa não corresponde à amostra anterior.")

linhas_elegiveis = amostra_completa.loc[passaram]
repetidas = linhas_elegiveis.duplicated(keep="first")

print("Colunas comparadas:", amostra_completa.shape[1])
print("Linhas que passaram no teste geográfico:", len(linhas_elegiveis))
print("Linhas adicionais integralmente repetidas:", int(repetidas.sum()))
print("Linhas que restariam:", int((~repetidas).sum()))

In [ ]:
from importlib import reload

reload(dados)

amostra_limpa, funil_a3 = dados.limpar_amostra_ssp(
    amostra_completa,
    retangulo=(-24.1, -23.3, -47.0, -46.3)
)

for etapa, quantidade in funil_a3.items():
    print(f"{etapa}: {quantidade}")

print("Dimensões finais:", amostra_limpa.shape)

In [ ]:
print("Lendo a aba JAN-JUN_2026 inteira...", flush=True)

tabela_jan_jun = dados.ler_ssp(
    raiz / "data/brutos/ssp/SPDadosCriminais_2026.xlsx",
    aba="JAN-JUN_2026"
)

print(f"Leitura concluída. Limpando {len(tabela_jan_jun):,} linhas...", flush=True)

jan_jun_limpa, funil_jan_jun = dados.limpar_amostra_ssp(
    tabela_jan_jun,
    retangulo=(-24.1, -23.3, -47.0, -46.3)
)

del tabela_jan_jun

for etapa, quantidade in funil_jan_jun.items():
    print(f"{etapa}: {quantidade}")

removidas = sum(
    quantidade
    for etapa, quantidade in funil_jan_jun.items()
    if etapa not in ("Linhas lidas", "Linhas finais")
)

print("Dimensões finais:", jan_jun_limpa.shape)
print(
    "O funil fechou?",
    funil_jan_jun["Linhas lidas"] == removidas + len(jan_jun_limpa)
)

In [ ]:
import pandas as pd

abas_restantes = {
    "SPDadosCriminais_2026.xlsx": ["JUL-DEZ_2026"],
    "CelularesSubtraidos_2026.xlsx": ["CELULAR_2026"],
    "ObjetosSubtraidos_2026.xlsx": [
        "OBJETOS_01_TRIM_2026",
        "OBJETOS_02_TRIM_2026",
        "OBJETOS_03_TRIM_2026",
    ],
}

pasta_ssp = raiz / "data" / "brutos" / "ssp"
retangulo_sp = (-24.1, -23.3, -47.0, -46.3)

resultados_a3 = [{
    "Arquivo": "SPDadosCriminais_2026.xlsx",
    "Aba": "JAN-JUN_2026",
    **funil_jan_jun,
}]

for arquivo, abas in abas_restantes.items():
    for aba in abas:
        print(f"Lendo {arquivo} — {aba}...", flush=True)

        tabela = dados.ler_ssp(pasta_ssp / arquivo, aba=aba)
        tabela_limpa, funil = dados.limpar_amostra_ssp(tabela, retangulo_sp)

        removidas = sum(
            quantidade
            for etapa, quantidade in funil.items()
            if etapa not in ("Linhas lidas", "Linhas finais")
        )

        if funil["Linhas lidas"] != removidas + len(tabela_limpa):
            raise ValueError(f"As contagens não fecharam na aba {aba}.")

        resultados_a3.append({"Arquivo": arquivo, "Aba": aba, **funil})
        del tabela, tabela_limpa

display(pd.DataFrame(resultados_a3))

In [ ]:
import pandas as pd

pasta_ssp = raiz / "data" / "brutos" / "ssp"
pasta_saida = raiz / "data" / "tratados"
pasta_saida.mkdir(parents=True, exist_ok=True)

csv_final = pasta_saida / "ssp_limpo.csv"
csv_temporario = pasta_saida / "ssp_limpo.csv.tmp"
retangulo_sp = (-24.1, -23.3, -47.0, -46.3)

campos_obrigatorios = [
    "nome_delegacia", "num_bo", "ano_bo", "data_ocorrencia_bo",
    "latitude", "longitude", "rubrica",
]
campos_complementares = [
    "natureza_apurada", "versao", "descr_tipolocal",
    "descr_subtipolocal", "bairro",
]
campos_saida = campos_obrigatorios + campos_complementares
colunas_csv = ["arquivo_origem", "aba_origem"] + campos_saida

resumo = pd.DataFrame(resultados_a3)
if len(resumo) != 6 or resumo.duplicated(["Arquivo", "Aba"]).any():
    raise ValueError("O diagnóstico precisa conter seis abas distintas.")

etapas_funil = [
    "Linhas lidas", "Sem data válida", "Sem coordenadas",
    "Coordenada igual a zero", "Fora do retângulo",
    "Linhas integralmente repetidas", "Linhas finais",
]

for numero, registro in enumerate(resultados_a3):
    arquivo = registro["Arquivo"]
    aba = registro["Aba"]
    print(f"Preparando {aba}...", flush=True)

    if (
        arquivo == "SPDadosCriminais_2026.xlsx"
        and aba == "JAN-JUN_2026"
        and "jan_jun_limpa" in globals()
    ):
        limpa, funil = jan_jun_limpa, funil_jan_jun
    else:
        bruta = dados.ler_ssp(pasta_ssp / arquivo, aba=aba)
        limpa, funil = dados.limpar_amostra_ssp(bruta, retangulo_sp)
        del bruta

    if any(funil[etapa] != registro[etapa] for etapa in etapas_funil):
        raise ValueError(f"O funil mudou em {aba}. Confira antes de exportar.")

    faltantes = set(campos_obrigatorios) - set(limpa.columns)
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes em {aba}: {faltantes}")

    saida = limpa.reindex(columns=campos_saida)
    saida.insert(0, "aba_origem", aba)
    saida.insert(0, "arquivo_origem", arquivo)

    saida.to_csv(
        csv_temporario,
        mode="w" if numero == 0 else "a",
        header=numero == 0,
        index=False,
        encoding="utf-8",
    )
    del limpa, saida

if pd.read_csv(csv_temporario, nrows=0).columns.tolist() != colunas_csv:
    raise ValueError("As colunas do CSV são diferentes das esperadas.")

linhas_conferidas = 0
for bloco in pd.read_csv(
    csv_temporario,
    usecols=["data_ocorrencia_bo", "latitude", "longitude"],
    chunksize=100_000,
):
    dentro = (
        bloco["latitude"].between(-24.1, -23.3)
        & bloco["longitude"].between(-47.0, -46.3)
    )
    if bloco["data_ocorrencia_bo"].isna().any() or not dentro.all():
        raise ValueError("O CSV contém data ausente ou coordenada fora do filtro.")
    linhas_conferidas += len(bloco)

total_esperado = int(resumo["Linhas finais"].sum())
if linhas_conferidas != total_esperado:
    raise ValueError("O total gravado no CSV não bate com o funil.")

csv_temporario.replace(csv_final)
resumo.to_csv(pasta_saida / "ssp_funil_a3.csv", index=False, encoding="utf-8")

print(f"CSV validado com {linhas_conferidas:,} linhas.")
print("Dados limpos:", csv_final)
print("Relatório do funil:", pasta_saida / "ssp_funil_a3.csv")

In [ ]:
%pip install folium

In [ ]:
import re
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pasta = raiz / "data" / "tratados"
entrada = pasta / "ssp_limpo.csv"
arquivo_limite = pasta / "limite_vila_mariana.geojson"
saida = pasta / "ssp_vila_mariana.csv"

for arquivo in [entrada, arquivo_limite]:
    if not arquivo.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

limite = gpd.read_file(arquivo_limite)
if (
    limite.empty
    or limite.crs is None
    or not limite.geom_type.isin(["Polygon", "MultiPolygon"]).all()
    or limite.geometry.is_empty.any()
    or not limite.geometry.is_valid.all()
):
    raise ValueError("Confira o polígono e o sistema de coordenadas do GeoJSON.")

limite = limite.to_crs("EPSG:4326")
poligono = limite.geometry.union_all()
oeste, sul, leste, norte = poligono.bounds

padrao_dp = r"\b0*(?:27|36)\s*[º°ª.]?\s*D\.?\s*P\.?\b"
total = total_dp = total_ambos = 0
partes = []
nomes_dp = set()

for bloco in pd.read_csv(entrada, dtype={"num_bo": "string"}, chunksize=100_000):
    total += len(bloco)
    dp = bloco["nome_delegacia"].str.contains(padrao_dp, flags=re.IGNORECASE, na=False)
    total_dp += int(dp.sum())
    nomes_dp.update(bloco.loc[dp, "nome_delegacia"].unique())

    candidatas = bloco.loc[
        bloco["longitude"].between(oeste, leste)
        & bloco["latitude"].between(sul, norte)
    ]
    pontos = gpd.GeoSeries(
        gpd.points_from_xy(candidatas["longitude"], candidatas["latitude"]),
        index=candidatas.index,
        crs="EPSG:4326",
    )
    dentro = pontos.within(poligono)
    selecionadas = candidatas.loc[dentro]
    partes.append(selecionadas)
    total_ambos += int(dp.loc[selecionadas.index].sum())

recorte = pd.concat(partes, ignore_index=True)

if recorte.empty:
    raise ValueError("O recorte ficou vazio. Confira o limite e as coordenadas.")

mapa = folium.Map()
folium.GeoJson(limite.to_json(), name="Limite da Vila Mariana").add_to(mapa)

amostra_mapa = recorte.sample(n=min(500, len(recorte)), random_state=42)
for lat, lon in amostra_mapa[["latitude", "longitude"]].itertuples(index=False):
    folium.CircleMarker(
        location=[lat, lon], radius=3, fill=True, fill_opacity=0.6
    ).add_to(mapa)

mapa.fit_bounds([[sul, oeste], [norte, leste]])
recorte.to_csv(saida, index=False, encoding="utf-8")

print(f"Linhas lidas da A3: {total:,}")
print(f"Dentro do polígono: {len(recorte):,}")
print(f"Registradas no 27º ou 36º DP: {total_dp:,}")
print(f"Atendem aos dois critérios: {total_ambos:,}")
print(f"Só pelo polígono: {len(recorte) - total_ambos:,}")
print(f"Só pela delegacia: {total_dp - total_ambos:,}")
print("Delegacias reconhecidas:", sorted(nomes_dp))
print("CSV do recorte:", saida)

display(mapa)

In [ ]:
arquivo_mapa = pasta / "mapa_conferencia_a4.html"
mapa.save(str(arquivo_mapa))
print(arquivo_mapa)

In [ ]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pasta = raiz / "data" / "tratados"

ocorrencias_vm = pd.read_csv(
    pasta / "ssp_vila_mariana.csv",
    dtype={"num_bo": "string", "rubrica": "string", "natureza_apurada": "string"},
)

datas = pd.to_datetime(
    ocorrencias_vm["data_ocorrencia_bo"], format="ISO8601", errors="raise"
).dt.normalize()

data_ref = datas.max()
print("Data mais recente no recorte:", data_ref.strftime("%d/%m/%Y"))
print("\nCinco datas mais recentes e suas contagens:")
print(datas.value_counts().sort_index(ascending=False).head(5).to_string())

for dias in [90, 180]:
    inicio = data_ref - pd.Timedelta(days=dias)
    quantidade = int(datas.between(inicio, data_ref).sum())
    print(f"Janela de {dias} dias desde {inicio:%d/%m/%Y}: {quantidade:,} linhas")

inicio = data_ref - pd.Timedelta(days=90)
if datas.between(inicio, data_ref).sum() < 30:
    inicio = data_ref - pd.Timedelta(days=180)

amostra_a5 = ocorrencias_vm.loc[datas.between(inicio, data_ref)]
categorias = amostra_a5["natureza_apurada"].str.strip().replace("", pd.NA)
categorias = categorias.fillna(amostra_a5["rubrica"].str.strip())

print(f"\nCategorias mais frequentes desde {inicio:%d/%m/%Y}:")
print(categorias.value_counts(dropna=False).head(20).to_string())
print("\nGrafo da Pessoa C disponível:", (pasta / "grafo_vila_mariana.graphml").is_file())

In [ ]:
chave_bo = ["nome_delegacia", "ano_bo", "num_bo"]

base = amostra_a5.copy()
base["categoria_diagnostico"] = categorias

print("Categorias por arquivo de origem:")
display(
    pd.crosstab(base["categoria_diagnostico"], base["arquivo_origem"])
    .reindex(categorias.value_counts().head(10).index, fill_value=0)
)

for coluna in chave_bo:
    base[coluna] = base[coluna].astype("string").str.strip().str.upper()

chave_completa = (
    base[chave_bo].fillna("").ne("").all(axis=1)
)

por_bo = (
    base.loc[chave_completa]
    .groupby(chave_bo)["arquivo_origem"]
    .agg(linhas="size", arquivos="nunique")
)

print("Linhas sem chave completa:", int((~chave_completa).sum()))
print("Chaves de BO distintas:", len(por_bo))
print("Chaves presentes em mais de um arquivo:", int(por_bo["arquivos"].gt(1).sum()))

In [ ]:
print("Linhas sem chave completa:", int((~chave_completa).sum()))
print("Chaves de BO distintas:", len(por_bo))
print("Chaves em mais de um arquivo:", int(por_bo["arquivos"].gt(1).sum()))

In [ ]:
categorias_piloto = {
    "FURTO - OUTROS": "Furto",
    "ROUBO - OUTROS": "Roubo",
    "LESÃO CORPORAL DOLOSA": "Lesão corporal dolosa",
}

piloto_a5 = base.loc[
    base["arquivo_origem"].eq("SPDadosCriminais_2026.xlsx")
].copy()

piloto_a5["categoria_piloto"] = (
    piloto_a5["categoria_diagnostico"]
    .str.strip()
    .str.upper()
    .map(categorias_piloto)
)

piloto_a5 = piloto_a5.dropna(subset=["categoria_piloto"])
piloto_a5["par_coordenadas"] = list(
    zip(piloto_a5["latitude"], piloto_a5["longitude"])
)

conferencia_a5 = piloto_a5.groupby(chave_bo).agg(
    linhas=("categoria_piloto", "size"),
    categorias=("categoria_piloto", "nunique"),
    locais=("par_coordenadas", "nunique"),
)

print("Linhas selecionadas:", len(piloto_a5))
print("Chaves de BO distintas:", len(conferencia_a5))
print("BOs com mais de uma categoria:", int(conferencia_a5["categorias"].gt(1).sum()))
print("BOs com mais de um par de coordenadas:", int(conferencia_a5["locais"].gt(1).sum()))

In [ ]:
piloto = piloto_a5.copy()
piloto["data_ocorrencia_bo"] = pd.to_datetime(
    piloto["data_ocorrencia_bo"], format="ISO8601", errors="raise"
).dt.normalize()

datas_por_bo = piloto.groupby(chave_bo)["data_ocorrencia_bo"].nunique()
datas_divergentes = int(datas_por_bo.ne(1).sum())

if piloto["data_ocorrencia_bo"].isna().any() or datas_divergentes:
    raise ValueError(
        f"Há datas ausentes ou {datas_divergentes} BOs com datas divergentes. "
        "Confira antes de consolidar."
    )

boletins_piloto = piloto.groupby(chave_bo, as_index=False).agg(
    data_ocorrencia_bo=("data_ocorrencia_bo", "first"),
    latitude=("latitude", "first"),
    longitude=("longitude", "first"),
    categorias=("categoria_piloto", lambda valores: " | ".join(sorted(set(valores)))),
    linhas_origem=("categoria_piloto", "size"),
)

if boletins_piloto["linhas_origem"].sum() != len(piloto):
    raise ValueError("A consolidação não contabilizou todas as linhas de origem.")

arquivo_piloto = pasta / "ssp_bo_piloto_a5.csv"
boletins_piloto.to_csv(arquivo_piloto, index=False, encoding="utf-8")

print(f"Piloto salvo com {len(boletins_piloto):,} boletins.")
print(arquivo_piloto)

In [ ]:
%pip install osmnx

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pasta = raiz / "data" / "tratados"
arquivo_grafo = pasta / "grafo_vila_mariana.graphml"
arquivo_boletins = pasta / "ssp_bo_piloto_a5.csv"

for arquivo in [arquivo_grafo, arquivo_boletins]:
    if not arquivo.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

# Carrega a rede de ruas e os boletins no mesmo sistema, em metros.
grafo = ox.io.load_graphml(arquivo_grafo)
if grafo.number_of_edges() == 0:
    raise ValueError("O grafo não contém trechos de rua.")

grafo_metros = ox.projection.project_graph(grafo, to_crs="EPSG:31983")

boletins = pd.read_csv(arquivo_boletins, dtype={"num_bo": "string"})
if boletins.empty or boletins[["latitude", "longitude"]].isna().any().any():
    raise ValueError("O piloto está vazio ou contém coordenadas ausentes.")

pontos = gpd.GeoSeries(
    gpd.points_from_xy(boletins["longitude"], boletins["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:31983")

arestas, distancias = ox.distance.nearest_edges(
    grafo_metros,
    X=pontos.x.to_numpy(),
    Y=pontos.y.to_numpy(),
    return_dist=True,
)

if not np.isfinite(distancias).all():
    raise ValueError("A busca retornou distâncias inválidas.")

boletins["edge_id"] = [f"{u}_{v}_{chave}" for u, v, chave in arestas]
boletins["distancia_trecho_m"] = distancias
associados = boletins.loc[boletins["distancia_trecho_m"].le(50)].copy()

if associados.empty:
    raise ValueError("Nenhum boletim ficou a até 50 m. Confira a área do grafo.")

# Começa com todos os trechos para preservar os que ficaram com zero.
trechos = pd.DataFrame([
    {
        "edge_id": f"{u}_{v}_{chave}",
        "comprimento_m": atributos.get("length"),
    }
    for u, v, chave, atributos in grafo.edges(keys=True, data=True)
])

trechos["comprimento_m"] = pd.to_numeric(trechos["comprimento_m"], errors="coerce")
if not np.isfinite(trechos["comprimento_m"]).all() or trechos["comprimento_m"].lt(0).any():
    raise ValueError("Há trechos com comprimento ausente ou inválido.")

contagem = associados["edge_id"].value_counts()
trechos["n_ocorrencias"] = trechos["edge_id"].map(contagem).fillna(0).astype(int)
trechos["ocorr_por_100m"] = (
    trechos["n_ocorrencias"] / (trechos["comprimento_m"].clip(lower=20) / 100)
)

if trechos["edge_id"].duplicated().any() or trechos["n_ocorrencias"].sum() != len(associados):
    raise ValueError("As chaves ou as contagens dos trechos não fecharam.")

associados.to_csv(pasta / "ocorrencias_recentes_piloto.csv", index=False)
trechos.to_csv(pasta / "ocorrencias_por_trecho_piloto.csv", index=False)

print(f"Boletins recebidos: {len(boletins)}")
print(f"Associados a até 50 m: {len(associados)}")
print(f"Fora do limite de 50 m: {len(boletins) - len(associados)}")
print(f"Trechos no grafo: {len(trechos)}")
print(f"Trechos com boletins associados: {int(trechos['n_ocorrencias'].gt(0).sum())}")
print("Arquivos do piloto salvos em:", pasta)

In [ ]:
from pathlib import Path

import folium
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

tratados = raiz / "data" / "tratados"
arquivo_grafo = tratados / "grafo_vila_mariana.graphml"
arquivo_luz = (
    raiz / "data" / "brutos" / "geosampa"
    / "SIRGAS_GPKG_iluminacaopublica.gpkg"
)
arquivo_trechos_a5 = tratados / "ocorrencias_por_trecho_piloto.csv"

for arquivo in [arquivo_grafo, arquivo_luz, arquivo_trechos_a5]:
    if not arquivo.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

raio_luz_m = 30
crs_metros = "EPSG:31983"

grafo_luz = ox.io.load_graphml(arquivo_grafo)
grafo_luz = ox.projection.project_graph(grafo_luz, to_crs=crs_metros)

trechos_luz = ox.convert.graph_to_gdfs(
    grafo_luz, nodes=False
)[["length", "geometry"]].reset_index()

trechos_luz["edge_id"] = (
    trechos_luz["u"].astype(str) + "_"
    + trechos_luz["v"].astype(str) + "_"
    + trechos_luz["key"].astype(str)
)
trechos_luz["comprimento_m"] = pd.to_numeric(
    trechos_luz["length"], errors="raise"
)

# Lê os pontos na área do grafo, com margem para os trechos da borda.
x_min, y_min, x_max, y_max = trechos_luz.total_bounds
luzes = gpd.read_file(
    arquivo_luz,
    layer="SIRGAS_GPKG_iluminacaopublica",
    bbox=(
        x_min - raio_luz_m, y_min - raio_luz_m,
        x_max + raio_luz_m, y_max + raio_luz_m,
    ),
)

if luzes.empty or luzes.crs is None or luzes.crs.to_epsg() != 31983:
    raise ValueError("Confira a cobertura e o sistema de coordenadas da iluminação.")

# Cada par é um ponto de iluminação a até 30 m de um trecho.
indices_trechos, indices_luzes = luzes.sindex.query(
    trechos_luz.geometry,
    predicate="dwithin",
    distance=raio_luz_m,
)

if len(indices_luzes) == 0:
    raise ValueError("Nenhum ponto foi associado. Confira as coordenadas.")

trechos_luz["n_luz"] = np.bincount(
    indices_trechos, minlength=len(trechos_luz)
)
trechos_luz["luz_por_100m"] = (
    trechos_luz["n_luz"]
    / (trechos_luz["comprimento_m"].clip(lower=20) / 100)
)

ids_a5 = pd.read_csv(arquivo_trechos_a5, usecols=["edge_id"])["edge_id"]

if (
    trechos_luz["edge_id"].duplicated().any()
    or set(trechos_luz["edge_id"]) != set(ids_a5)
):
    raise ValueError("Os trechos da iluminação não batem com os da A5.")

saida_luz = tratados / "iluminacao_por_trecho.csv"
trechos_luz[["edge_id", "n_luz", "luz_por_100m"]].to_csv(
    saida_luz, index=False
)

# Monta o mapa com as ruas e uma amostra dos pontos associados.
ruas_mapa = trechos_luz[["n_luz", "geometry"]].to_crs("EPSG:4326")
pontos_mapa = luzes.iloc[np.unique(indices_luzes)].to_crs("EPSG:4326")
pontos_mapa = pontos_mapa.sample(
    n=min(1_000, len(pontos_mapa)), random_state=42
)

mapa_luz = folium.Map()
folium.GeoJson(
    ruas_mapa.to_json(),
    style_function=lambda _: {"color": "#2563eb", "weight": 2},
    tooltip=folium.GeoJsonTooltip(
        fields=["n_luz"],
        aliases=["Pontos cadastrados a até 30 m"],
    ),
).add_to(mapa_luz)

for ponto in pontos_mapa.geometry:
    folium.CircleMarker(
        [ponto.y, ponto.x],
        radius=2,
        color="#d97706",
        fill=True,
        fill_opacity=0.8,
    ).add_to(mapa_luz)

oeste, sul, leste, norte = ruas_mapa.total_bounds
mapa_luz.fit_bounds([[sul, oeste], [norte, leste]])
arquivo_mapa = tratados / "mapa_conferencia_a6.html"
mapa_luz.save(str(arquivo_mapa))

print(f"Trechos avaliados: {len(trechos_luz)}")
print(f"Trechos com pelo menos um ponto: {int(trechos_luz['n_luz'].gt(0).sum())}")
print(f"Pontos distintos associados: {len(np.unique(indices_luzes))}")
print("CSV:", saida_luz)
print("Mapa:", arquivo_mapa)
print("Arquivo da Pessoa B disponível:", (tratados / "abrigos.csv").is_file())

In [ ]:
arquivo_abrigos = tratados / "abrigos.csv"
abrigos_disponiveis = arquivo_abrigos.is_file()

apoio = trechos_luz[["edge_id"]].copy()
apoio["n_abrigos"] = 0

if abrigos_disponiveis:
    abrigos = pd.read_csv(arquivo_abrigos)
    faltantes = {"lat", "lon"} - set(abrigos.columns)
    if faltantes:
        raise ValueError(f"Colunas ausentes em abrigos.csv: {sorted(faltantes)}")
    if abrigos.empty:
        raise ValueError("abrigos.csv está vazio. Confira a entrega da Pessoa B.")

    lat = pd.to_numeric(abrigos["lat"], errors="coerce")
    lon = pd.to_numeric(abrigos["lon"], errors="coerce")
    if not (lat.between(-90, 90) & lon.between(-180, 180)).all():
        raise ValueError("Há coordenadas ausentes ou inválidas em abrigos.csv.")

    pontos_apoio = gpd.GeoSeries(
        gpd.points_from_xy(lon, lat), crs="EPSG:4326"
    ).to_crs(trechos_luz.crs)

    idx_trechos, _ = pontos_apoio.sindex.query(
        trechos_luz.geometry, predicate="dwithin", distance=100
    )
    if len(idx_trechos) == 0:
        raise ValueError("Nenhum abrigo ficou a até 100 m. Confira as coordenadas.")

    apoio["n_abrigos"] = np.bincount(
        idx_trechos, minlength=len(trechos_luz)
    )

apoio["abrigos_disponiveis"] = abrigos_disponiveis

ocorrencias = pd.read_csv(tratados / "ocorrencias_por_trecho_piloto.csv")
iluminacao = pd.read_csv(tratados / "iluminacao_por_trecho.csv")

ids_grafo = set(trechos_luz["edge_id"])
for nome, tabela in [
    ("ocorrências", ocorrencias),
    ("iluminação", iluminacao),
    ("locais de apoio", apoio),
]:
    if tabela["edge_id"].duplicated().any() or set(tabela["edge_id"]) != ids_grafo:
        raise ValueError(f"As chaves de {nome} não batem com as do grafo.")

indicadores = (
    ocorrencias
    .merge(iluminacao, on="edge_id", validate="one_to_one")
    .merge(apoio, on="edge_id", validate="one_to_one")
)

if len(indicadores) != len(ids_grafo) or indicadores.isna().any().any():
    raise ValueError("A tabela integrada tem linhas faltantes ou valores ausentes.")

saida_indicadores = tratados / "indicadores_por_trecho_piloto.csv"
indicadores.to_csv(saida_indicadores, index=False, encoding="utf-8")

print("Trechos na tabela integrada:", len(indicadores))
print("Trechos com locais de apoio:", int(indicadores["n_abrigos"].gt(0).sum()))
print("Dados da Pessoa B disponíveis:", abrigos_disponiveis)
print("CSV:", saida_indicadores)

In [ ]:
from pathlib import Path

import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

tratados = raiz / "data" / "tratados"
arquivo_hotspots = tratados / "indicadores_com_hotspots_piloto.csv"

indicadores = pd.read_csv(tratados / "indicadores_por_trecho_piloto.csv")
marcacao = pd.read_csv(arquivo_hotspots, usecols=["edge_id", "em_hotspot"])

apoio_disponivel = indicadores["abrigos_disponiveis"].astype(str).str.lower()
if indicadores.empty or not apoio_disponivel.eq("true").all():
    raise ValueError("A tabela da A7/A8 ainda está sem os dados de apoio.")

if set(indicadores["edge_id"]) != set(marcacao["edge_id"]):
    raise ValueError("Os trechos da A8 e da A9 são diferentes.")

atualizados = indicadores.merge(marcacao, on="edge_id", validate="one_to_one")
atualizados.to_csv(arquivo_hotspots, index=False, encoding="utf-8")

print("Trechos atualizados:", len(atualizados))
print("Trechos com locais de apoio:", int(atualizados["n_abrigos"].gt(0).sum()))
print("Trechos em hotspot:", int(atualizados["em_hotspot"].eq(1).sum()))

In [ ]:
%pip install scikit-learn

In [ ]:
from itertools import product
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from sklearn.cluster import DBSCAN

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

tratados = raiz / "data" / "tratados"
arquivo = tratados / "ocorrencias_recentes_piloto.csv"
boletins = pd.read_csv(arquivo, usecols=["latitude", "longitude"])

if boletins.empty or boletins.isna().any().any():
    raise ValueError("O CSV do piloto está vazio ou contém coordenadas ausentes.")

pontos = gpd.GeoSeries(
    gpd.points_from_xy(boletins["longitude"], boletins["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:31983")

xy = pd.DataFrame({"x": pontos.x, "y": pontos.y})

mapa_a9 = folium.Map()
camada_pontos = folium.FeatureGroup(name="Boletins do piloto", show=True)

for lat, lon in boletins.itertuples(index=False, name=None):
    folium.CircleMarker(
        location=[lat, lon],
        radius=2,
        weight=0,
        fill=True,
        fill_color="#334155",
        fill_opacity=0.55,
    ).add_to(camada_pontos)

camada_pontos.add_to(mapa_a9)
resultados = []

# Margem visual provisória, aplicada após o agrupamento.
margem_mapa_m = 15

for raio, minimo in product([100, 150, 200, 300], [3, 5, 8]):
    rotulos = DBSCAN(eps=raio, min_samples=minimo).fit_predict(xy)
    agrupados = xy.assign(grupo=rotulos).query("grupo >= 0")
    tamanhos = agrupados["grupo"].value_counts()

    resultados.append({
        "eps_m": raio,
        "min_samples": minimo,
        "grupos": len(tamanhos),
        "ruido_pct": round(100 * (rotulos == -1).mean(), 1),
        "maior_grupo": int(tamanhos.max()) if not tamanhos.empty else 0,
    })

    if tamanhos.empty:
        continue

    formas = []
    for grupo, linhas in agrupados.groupby("grupo"):
        forma = MultiPoint(linhas[["x", "y"]].to_numpy())
        formas.append({
            "grupo": int(grupo),
            "boletins": len(linhas),
            "geometry": forma.convex_hull.buffer(margem_mapa_m),
        })

    poligonos = gpd.GeoDataFrame(formas, crs="EPSG:31983").to_crs("EPSG:4326")
    folium.GeoJson(
        poligonos.to_json(),
        name=f"{raio} m | mínimo {minimo}",
        show=(raio, minimo) == (150, 5),
        style_function=lambda _: {
            "color": "#b45309", "weight": 2, "fillOpacity": 0.22,
        },
        tooltip=folium.GeoJsonTooltip(fields=["grupo", "boletins"]),
    ).add_to(mapa_a9)

mapa_a9.fit_bounds([
    [boletins["latitude"].min(), boletins["longitude"].min()],
    [boletins["latitude"].max(), boletins["longitude"].max()],
])
folium.LayerControl(collapsed=False).add_to(mapa_a9)

resumo_a9 = pd.DataFrame(resultados)
resumo_a9.to_csv(tratados / "calibracao_hotspots_piloto.csv", index=False)
mapa_a9.save(str(tratados / "mapa_calibracao_a9.html"))

print(f"Boletins analisados: {len(boletins)}")
print(resumo_a9.to_string(index=False))
print("Mapa:", tratados / "mapa_calibracao_a9.html")

In [ ]:
from pathlib import Path

import geopandas as gpd
import osmnx as ox
import pandas as pd
from shapely.geometry import MultiPoint
from sklearn.cluster import DBSCAN

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

tratados = raiz / "data" / "tratados"
crs_metros = "EPSG:31983"

boletins = pd.read_csv(
    tratados / "ocorrencias_recentes_piloto.csv",
    usecols=["latitude", "longitude"],
)

pontos = gpd.GeoSeries(
    gpd.points_from_xy(boletins["longitude"], boletins["latitude"]),
    crs="EPSG:4326",
).to_crs(crs_metros)

xy = pd.DataFrame({"x": pontos.x, "y": pontos.y})
xy["grupo"] = DBSCAN(eps=100, min_samples=5).fit_predict(xy[["x", "y"]])

# A margem de 15 m desenha o polígono após o agrupamento.
formas = []
for grupo, linhas in xy.loc[xy["grupo"].ge(0)].groupby("grupo"):
    envoltoria = MultiPoint(linhas[["x", "y"]].to_numpy()).convex_hull
    formas.append({
        "grupo": int(grupo),
        "n_boletins": len(linhas),
        "geometry": envoltoria.buffer(15),
    })

if not formas:
    raise ValueError("O DBSCAN não gerou grupos. Confira os dados do piloto.")

hotspots = gpd.GeoDataFrame(formas, crs=crs_metros)

grafo = ox.io.load_graphml(tratados / "grafo_vila_mariana.graphml")
grafo_metros = ox.projection.project_graph(grafo, to_crs=crs_metros)
trechos = ox.convert.graph_to_gdfs(grafo_metros, nodes=False)

marcacao = pd.DataFrame({
    "edge_id": [f"{u}_{v}_{chave}" for u, v, chave in trechos.index],
    "em_hotspot": trechos.geometry.intersects(
        hotspots.geometry.union_all()
    ).astype(int).to_numpy(),
})

indicadores = pd.read_csv(tratados / "indicadores_por_trecho_piloto.csv")
if set(marcacao["edge_id"]) != set(indicadores["edge_id"]):
    raise ValueError("As chaves do grafo não batem com as da tabela da A8.")

com_hotspots = indicadores.merge(marcacao, on="edge_id", validate="one_to_one")

hotspots.to_crs("EPSG:4326").to_file(
    tratados / "hotspots_piloto.geojson", driver="GeoJSON", index=False
)
com_hotspots.to_csv(tratados / "indicadores_com_hotspots_piloto.csv", index=False)

total_em_hotspot = int(com_hotspots["em_hotspot"].sum())
print(f"Hotspots gerados: {len(hotspots)}")
print(f"Trechos avaliados: {len(com_hotspots)}")
print(
    f"Trechos em hotspot: {total_em_hotspot} "
    f"({total_em_hotspot / len(com_hotspots):.1%})"
)

In [ ]:
from pathlib import Path

import numpy as np
import osmnx as ox
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

tratados = raiz / "data" / "tratados"
indicadores = pd.read_csv(tratados / "indicadores_com_hotspots_piloto.csv")

# Pesos e limites iniciais do guia.
pesos = {"luz": 0.35, "ocorrencias": 0.35, "abrigos": 0.15, "hotspot": 0.15}
limite_verde, limite_amarelo = 70, 40

# O método average dá a mesma posição relativa a valores empatados.
indicadores["score_iluminacao"] = (
    100 * indicadores["luz_por_100m"].rank(pct=True, method="average")
)
nota_ocorrencias = (
    100 * (1 - indicadores["ocorr_por_100m"].rank(pct=True, method="average"))
)
nota_abrigos = 50 * indicadores["n_abrigos"].clip(upper=2)
nota_hotspot = 100 * (1 - indicadores["em_hotspot"])

indicadores["score_base"] = (
    pesos["luz"] * indicadores["score_iluminacao"]
    + pesos["ocorrencias"] * nota_ocorrencias
    + pesos["abrigos"] * nota_abrigos
    + pesos["hotspot"] * nota_hotspot
)
indicadores["cor"] = np.select(
    [
        indicadores["score_base"].ge(limite_verde),
        indicadores["score_base"].ge(limite_amarelo),
    ],
    ["verde", "amarelo"],
    default="vermelho",
)

# Obtém o nome e o ponto médio de cada trecho no grafo da Pessoa C.
grafo = ox.io.load_graphml(tratados / "grafo_vila_mariana.graphml")
grafo_metros = ox.projection.project_graph(grafo, to_crs="EPSG:31983")
arestas = ox.convert.graph_to_gdfs(grafo_metros, nodes=False)
meios = arestas.geometry.interpolate(0.5, normalized=True).to_crs("EPSG:4326")

nomes = [
    " / ".join(map(str, nome)) if isinstance(nome, list)
    else "" if pd.isna(nome) else str(nome)
    for nome in arestas.get("name", pd.Series("", index=arestas.index))
]
dados_ruas = pd.DataFrame({
    "edge_id": [f"{u}_{v}_{chave}" for u, v, chave in arestas.index],
    "nome_rua": nomes,
    "lat_meio": meios.y.to_numpy(),
    "lon_meio": meios.x.to_numpy(),
})

if set(indicadores["edge_id"]) != set(dados_ruas["edge_id"]):
    raise ValueError("Os trechos dos indicadores não batem com os do grafo.")

colunas = [
    "edge_id", "nome_rua", "comprimento_m", "lat_meio", "lon_meio",
    "n_luz", "n_ocorrencias", "n_abrigos", "em_hotspot",
    "score_iluminacao", "score_base", "cor",
]
saida = indicadores.merge(dados_ruas, on="edge_id", validate="one_to_one")[colunas]

notas = saida[["score_iluminacao", "score_base"]]
if saida.isna().any().any() or not (notas.ge(0) & notas.le(100)).all().all():
    raise ValueError("O resultado tem campos ausentes ou scores fora de 0 a 100.")

arquivo_saida = tratados / "trechos_score_piloto.csv"
saida.to_csv(arquivo_saida, index=False, encoding="utf-8")

cores = saida["cor"].value_counts().reindex(
    ["verde", "amarelo", "vermelho"], fill_value=0
)
print("Trechos no CSV:", len(saida))
print(f"Score mínimo: {saida['score_base'].min():.2f}")
print(f"Score máximo: {saida['score_base'].max():.2f}")
for cor, total in cores.items():
    print(f"{cor}: {total} trechos ({total / len(saida):.1%})")
print("CSV do piloto:", arquivo_saida)

In [ ]:
from pathlib import Path
from shutil import copyfile

import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pasta = raiz / "data" / "tratados"
origem = pasta / "trechos_score_piloto.csv"
destino = pasta / "trechos_score.csv"

score = pd.read_csv(origem).set_index("edge_id", verify_integrity=True)
indicadores = pd.read_csv(
    pasta / "indicadores_por_trecho_piloto.csv"
).set_index("edge_id", verify_integrity=True)

if len(score) != 12134 or set(score.index) != set(indicadores.index):
    raise ValueError("Os trechos do score não correspondem aos da A7/A8.")

if not indicadores["abrigos_disponiveis"].astype(str).str.lower().eq("true").all():
    raise ValueError("A tabela da A7/A8 ainda está sem os dados de apoio.")

if not score["n_abrigos"].eq(indicadores["n_abrigos"].reindex(score.index)).all():
    raise ValueError("O score ainda usa uma contagem antiga de locais de apoio.")

if set(score["cor"]) != {"verde", "amarelo", "vermelho"}:
    raise ValueError("O score precisa conter as três cores.")

temporario = destino.with_suffix(".csv.tmp")
copyfile(origem, temporario)
temporario.replace(destino)

print("CSV pronto para conferência:", destino)

In [ ]:
import math
import sys
from pathlib import Path

import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.score import avaliar_rota

pasta = raiz / "data" / "tratados"
arquivo_score = pasta / "trechos_score.csv"
if not arquivo_score.is_file():
    arquivo_score = pasta / "trechos_score_piloto.csv"

scores = pd.read_csv(arquivo_score)
trecho = scores.loc[scores["score_base"].idxmax()]
edge_id = trecho["edge_id"]

alertas_teste = pd.DataFrame({
    "id": range(1, 6),
    "lat": [trecho["lat_meio"]] * 5,
    "lon": [trecho["lon_meio"]] * 5,
})

for quantidade, desconto in [(0, 0), (1, 10), (5, 30)]:
    resultado = avaliar_rota([edge_id], alertas_teste.head(quantidade))
    esperado = max(0, trecho["score_base"] - desconto)

    assert resultado["n_alertas_na_rota"] == quantidade
    assert resultado["penalidade_alertas"] == desconto
    assert resultado["pior_trecho"] == edge_id
    assert resultado["n_ocorrencias"] == trecho["n_ocorrencias"]
    assert math.isclose(resultado["score_medio"], esperado, abs_tol=1e-9)

    print(
        f"{quantidade} alerta(s) | desconto: {desconto} | "
        f"score: {resultado['score_medio']:.2f} | cor: {resultado['cor']}"
    )

In [ ]:
import osmnx as ox

grafo_teste = ox.io.load_graphml(pasta / "grafo_vila_mariana.graphml")
scores_por_trecho = scores.set_index("edge_id")

def escolher_rota_teste():
    for u, v, chave_a in grafo_teste.edges(keys=True):
        for _, w, chave_b in grafo_teste.out_edges(v, keys=True):
            if len({u, v, w}) != 3:
                continue

            ids = [f"{u}_{v}_{chave_a}", f"{v}_{w}_{chave_b}"]
            if not all(edge_id in scores_por_trecho.index for edge_id in ids):
                continue

            trechos = scores_por_trecho.loc[ids]
            if (
                trechos["comprimento_m"].gt(0).all()
                and trechos["comprimento_m"].nunique() == 2
                and trechos["score_base"].nunique() == 2
            ):
                return ids, trechos

    raise ValueError("Nenhuma rota de dois trechos adequada ao teste foi encontrada.")

ids, trechos = escolher_rota_teste()
a, b = trechos.iloc[0], trechos.iloc[1]

esperado = (
    a["score_base"] * a["comprimento_m"]
    + b["score_base"] * b["comprimento_m"]
) / (a["comprimento_m"] + b["comprimento_m"])

resultado = avaliar_rota(ids, alertas_teste.head(0))

assert math.isclose(resultado["score_medio"], esperado, abs_tol=1e-9)
assert resultado["pior_trecho"] == trechos["score_base"].idxmin()
assert resultado["n_ocorrencias"] == trechos["n_ocorrencias"].sum()
assert resultado["n_trechos_com_luz"] == trechos["n_luz"].gt(0).sum()
assert resultado["n_trechos_em_hotspot"] == trechos["em_hotspot"].eq(1).sum()

print(f"Média simples dos dois trechos: {trechos['score_base'].mean():.4f}")
print(f"Média ponderada esperada: {esperado:.4f}")
print(f"Média retornada pela função: {resultado['score_medio']:.4f}")
print("Teste da rota com dois trechos passou.")

In [ ]:
from pathlib import Path

import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pasta = raiz / "data" / "tratados"
arquivo = pasta / "trechos_score.csv"
if not arquivo.is_file():
    arquivo = pasta / "trechos_score_piloto.csv"

trechos = pd.read_csv(arquivo)
nomes = trechos["nome_rua"].fillna("").str.split(" / ")

ruas = [
    "Rua Domingos de Morais",
    "Rua Vergueiro",
    "Rua Joaquim Távora",
    "Rua Sena Madureira",
    "Rua Humberto I",
]

resumo = []

for rua in ruas:
    corresponde = nomes.apply(
        lambda lista: any(nome.strip().casefold() == rua.casefold() for nome in lista)
    )
    partes = trechos.loc[corresponde]

    if partes.empty:
        print(f"Nome não encontrado no grafo: {rua}")
        continue

    resumo.append({
        "Rua": rua,
        "Trechos": len(partes),
        "Menor score": round(partes["score_base"].min(), 2),
        "Maior score": round(partes["score_base"].max(), 2),
        "Verdes": int(partes["cor"].eq("verde").sum()),
        "Amarelos": int(partes["cor"].eq("amarelo").sum()),
        "Vermelhos": int(partes["cor"].eq("vermelho").sum()),
        "Trechos com iluminação cadastrada": int(partes["n_luz"].gt(0).sum()),
        "Trechos em hotspot": int(partes["em_hotspot"].eq(1).sum()),
    })

print("Arquivo avaliado:", arquivo.name)
print(pd.DataFrame(resumo).to_string(index=False))